In [15]:
# '##' marks changes from the main unablated version

# Import

In [1]:
import os
import time
import json
import random
import pickle
import numpy as np
import pandas as pd
import tifffile as tiff

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import Sampler,Dataset,DataLoader,random_split
from pytorch_metric_learning.losses import ArcFaceLoss,SubCenterArcFaceLoss

from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score,f1_score

import warnings
warnings.filterwarnings("ignore",category=FutureWarning)

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Model components

In [2]:
class BasicBlock(nn.Module):
    expansion=1
    def __init__(self,in_channel,out_channel,stride=1,downsample=None,**kwargs):
        super(BasicBlock,self).__init__()
        self.conv1=nn.Conv2d(in_channels=in_channel,out_channels=out_channel,
                               kernel_size=3,stride=stride,padding=1,bias=False)
        self.bn1=nn.BatchNorm2d(out_channel)
        self.relu=nn.ReLU()
        self.conv2=nn.Conv2d(in_channels=out_channel,out_channels=out_channel,
                               kernel_size=3,stride=1,padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(out_channel)
        self.downsample=downsample

    def forward(self,x):
        identity=x
        if self.downsample is not None:
            identity=self.downsample(x)

        out=self.conv1(x)
        out=self.bn1(out)
        out=self.relu(out)

        out=self.conv2(out)
        out=self.bn2(out)

        out+=identity
        out=self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self,block,blocks_num,data_channel,
                 num_classes=1000,include_top=True,groups=1,width_per_group=64):
        super(ResNet,self).__init__()
        self.data_channel=data_channel
        self.include_top=include_top
        self.in_channel=64

        self.groups=groups
        self.width_per_group=width_per_group

        self.conv1=nn.Conv2d(data_channel,self.in_channel,kernel_size=7,stride=2,padding=3,bias=False)
        self.bn1=nn.BatchNorm2d(self.in_channel)
        self.relu=nn.ReLU(inplace=True)
        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2,padding=1)
        self.layer1=self._make_layer(block,64,blocks_num[0])
        self.layer2=self._make_layer(block,128,blocks_num[1],stride=2)
        self.layer3=self._make_layer(block,256,blocks_num[2],stride=2)
        self.layer4=self._make_layer(block,512,blocks_num[3],stride=2)
        if self.include_top:
            self.avgpool=nn.AdaptiveAvgPool2d((1,1))  # output size=(1,1)
            self.fc=nn.Linear(512 * block.expansion,num_classes)

        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')

    def _make_layer(self,block,channel,block_num,stride=1):
        downsample=None
        if stride!=1 or self.in_channel!=channel * block.expansion:
            downsample=nn.Sequential(
                nn.Conv2d(self.in_channel,channel * block.expansion,kernel_size=1,stride=stride,bias=False),
                nn.BatchNorm2d(channel * block.expansion))

        layers=[]
        layers.append(block(self.in_channel,channel,
                            downsample=downsample,stride=stride,
                            groups=self.groups,width_per_group=self.width_per_group))
        self.in_channel=channel * block.expansion

        for _ in range(1,block_num):
            layers.append(block(self.in_channel,channel,
                                groups=self.groups,width_per_group=self.width_per_group))
        return nn.Sequential(*layers)

    def forward(self,x):
        x=self.conv1(x)
        x=self.bn1(x)
        x=self.relu(x)
        x=self.maxpool(x)

        x=self.layer1(x)
        x=self.layer2(x)
        x=self.layer3(x)
        x=self.layer4(x)

        if self.include_top:
            x=self.avgpool(x)
            x=torch.flatten(x, 1)
            x=self.fc(x)
        return x

def resnet34(data_channel,num_classes=1000,include_top=True):
    return ResNet(BasicBlock,[3,4,6,3],data_channel,num_classes=num_classes,include_top=include_top)

In [3]:
class CrossAttention(nn.Module):
    def __init__(self,feat_dim,pos_dim):
        super().__init__()
        self.query=nn.Conv2d(feat_dim,feat_dim,1)
        self.key=nn.Conv2d(pos_dim,feat_dim,1)
        self.value=nn.Conv2d(pos_dim,feat_dim,1)

        self.alpha=nn.Parameter(torch.ones(1))
    def forward(self,feat,pos):
        Q=self.query(feat)  #[B,C,H,W]
        K=self.key(pos)     #[B,C,H,W]
        V=self.value(pos)   #[B,C,H,W]

        attn=torch.einsum('bchw,bchw->bhw',Q,K)  #[B,H,W]
        attn=torch.sigmoid(attn).unsqueeze(1)  #[B,1,H,W]
        return feat+self.alpha*(attn*V)

# Model

In [4]:
class DualStreamPositionNet(nn.Module):
    def __init__(self,feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate=0):
        super().__init__()
        self.feat_conv=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,7,padding=3),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU()
        )
        self.shortcut=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,1),
            nn.BatchNorm2d(hid_dim)
        )

        self.cross_attn=CrossAttention(hid_dim,pos_dim)
        self.ResNet=resnet34(hid_dim)        
        self.ResNet.fc=nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(self.ResNet.fc.in_features,emb_dim,bias=False)
        )

    def forward(self,feat,pos):
        residual=self.shortcut(feat)
        feat=self.feat_conv(feat)
        feat=feat+residual

        fused=self.cross_attn(feat,pos)
        embedding=self.ResNet(fused)
        embedding=nn.functional.normalize(embedding,p=2,dim=-1)
        return embedding

# TiffDataset

In [5]:
class TiffDataset(Dataset):
    def __init__(self,root_dir,background_dir,embedding_type,transform=None,classes=None):
        self.root_dir=root_dir
        self.background_dir=background_dir
        self.transform=transform
        self.samples=[]
        if classes is not None:
            self.classes=classes
        else:
            self.classes=self._find_classes(self.root_dir)

        for class_name in self.classes:
            class_dir=os.path.join(root_dir,str(class_name))
            if os.path.isdir(class_dir):
                for fname in os.listdir(class_dir):
                    if fname.endswith(".tiff") or fname.endswith(".tif"):
                        self.samples.append((os.path.join(class_dir,fname),class_name))

        self.background_emb=tiff.imread(os.path.join(background_dir,f'{embedding_type}_background_emb.tiff')).astype('float32')
        self.background_emb=torch.tensor(self.background_emb)

        self.background_pos=tiff.imread(os.path.join(background_dir,f'{embedding_type}_background_pos.tiff')).astype('float32')
        self.background_pos=torch.tensor(self.background_pos)
        x,y=self.background_pos[0],self.background_pos[1]
        self.min_x=torch.min(x)
        self.max_x=torch.max(x)
        self.min_y=torch.min(y)
        self.max_y=torch.max(y)

    def _find_classes(self,root_dir):
        classes=sorted([int(d.name) for d in os.scandir(root_dir) if d.is_dir()])
        return classes

    def __len__(self):
        return len(self.samples)

    def _pos_encoding(self,pos):
        x,y=pos[0],pos[1]
        mask=(x==0)&(y==0)
        x=(x-self.min_x)/(self.max_x-self.min_x)
        y=(y-self.min_y)/(self.max_y-self.min_y)

        num_freq=8
        div_term=torch.exp(torch.arange(0,num_freq)*(-np.log(10000.0)/num_freq))
        x_enc=torch.cat([torch.sin(x.unsqueeze(-1)*div_term),torch.cos(x.unsqueeze(-1)*div_term)],dim=-1).permute(2,0,1)
        y_enc=torch.cat([torch.sin(y.unsqueeze(-1)*div_term),torch.cos(y.unsqueeze(-1)*div_term)],dim=-1).permute(2,0,1)

        combined_enc=torch.cat([x_enc,y_enc],dim=0)
        combined_enc=combined_enc.masked_fill(mask.unsqueeze(0),0)
        return combined_enc

    def __getitem__(self,idx):
        file_path,label=self.samples[idx]
        image=tiff.imread(file_path).astype('float32')
        image=torch.tensor(image)

        comb_channels=torch.cat((self.background_pos,self.background_emb,image),dim=0)
        if self.transform:
            comb_channels=self.transform(comb_channels)

        feature_channels=comb_channels[self.background_pos.shape[0]:]
        pos_channels=comb_channels[:self.background_pos.shape[0]]

        pos_channels=self._pos_encoding(pos_channels)
        return {'feature_channels':feature_channels,'pos_channels':pos_channels,'label':torch.tensor(label)}

In [6]:
def set_interpolation(transform_list,mode):
    new_transforms=[]
    for t in transform_list:
        if hasattr(t,'interpolation'):
            t.interpolation=mode
        new_transforms.append(t)
    return transforms.Compose(new_transforms)

data_transform={
    'train': transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.3,1.0)),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.RandomVerticalFlip(), # theoretically also justified
                                 ##transforms.RandomRotation(30), # would cause much slower convergence
                                ]),
    'val': transforms.Compose([transforms.Resize(256),
                                transforms.CenterCrop(224),
                              ]),
    'test': transforms.Compose([transforms.Resize(256),
                                transforms.CenterCrop(224),])}

for trans_name,trans in data_transform.items():
    data_transform[trans_name]=set_interpolation(trans.transforms,InterpolationMode.BILINEAR)

# For train and test

In [7]:
def test(model,test_loaders,loss_function,device):
    to_return_metrics={'accuracy':{},'f1_score':{}}
    model.eval()
    loss_function.eval()
    with torch.no_grad():
        for species,test_loader in test_loaders:
            all_labels,all_preds,all_probs=[],[],[]
            for data in tqdm(test_loader,desc=f"Testing {species}",leave=False):
                data={k:v.to(torch.float32).to(device) for k, v in data.items()}
                labels=data['label'].to(torch.int64)

                embeddings=model(data['feature_channels'],data['pos_channels'])
                logits=loss_function[species].get_cosine(embeddings)*loss_function[species].scale
                probs=torch.softmax(logits,dim=1)
                _,preds=torch.max(probs,1)

                all_labels.append(labels.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
            all_labels=np.concatenate(all_labels)
            all_preds=np.concatenate(all_preds)
            all_probs=np.concatenate(all_probs)

            accuracy=accuracy_score(all_labels,all_preds)
            f1=f1_score(all_labels,all_preds,average='weighted')

            to_return_metrics['accuracy'][species]=round(accuracy,4)
            to_return_metrics['f1_score'][species]=round(f1,4)

            print(f'Species: {species}')
            print('accuracy:{:.4f}, f1_score:{:.4f}'.format(accuracy,f1))
        torch.cuda.empty_cache()
        return to_return_metrics

In [8]:
def train(model,train_loaders,val_loaders,loss_function,optimizer,device,epochs):
    metrics_by_epoch={}
    torch.cuda.empty_cache()
    for epoch in range(epochs):
        total_loss=0
        model.train()
        loss_function.train()

        species_list=[species for species,dl in train_loaders]
        iterators=[iter(dl) for species,dl in train_loaders]
        pseudo_combined_batch_steps=len(species_list)

        total_steps=sum([len(dl) for species,dl in train_loaders])
        progress_bar=tqdm(total=total_steps,desc=f"Epoch {epoch+1}")
        i=0
        while True:
            selected_idx=i% pseudo_combined_batch_steps
            try:
                data=next(iterators[selected_idx])
            except StopIteration:
                break

            species=species_list[selected_idx]
            data={k:v.to(torch.float32).to(device) for k,v in data.items()}
            labels=data['label'].to(torch.int64)
            embeddings=model(data['feature_channels'],data['pos_channels'])
            loss=loss_function[species](embeddings,labels)/pseudo_combined_batch_steps
            loss.backward()

            i+=1
            if i%pseudo_combined_batch_steps==0:
                optimizer.step()
                optimizer.zero_grad()
            total_loss+=loss.item()*pseudo_combined_batch_steps
            progress_bar.update(1)
            progress_bar.set_postfix(loss=total_loss/progress_bar.n)
        progress_bar.close()
        print('epoch {}, loss:{:.4f}'.format(epoch+1,total_loss/total_steps))

        if epoch%5==4:
            print('At epoch '+str(epoch+1),':')
            metrics_by_epoch[epoch+1]=test(model,val_loaders,loss_function,device)
            torch.save(model.state_dict(),'./model/model_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')
            torch.save(loss_function.state_dict(),'./model/loss_func_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')
    with open('./model/metrics_by_epoch_'+str(int(time.time()))+'.json','w') as f:
        json.dump(metrics_by_epoch,f,indent=4)

# Train and Test without using background channel

## changes

In [9]:
class DualStreamPositionNet_ablate_V1(nn.Module):
    def __init__(self,feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate=0):
        super().__init__()
        self.feat_conv=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,7,padding=3),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU()
        )
        self.shortcut=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,1),
            nn.BatchNorm2d(hid_dim)
        )

        self.cross_attn=CrossAttention(hid_dim,pos_dim)
        self.ResNet=resnet34(hid_dim)        
        self.ResNet.fc=nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(self.ResNet.fc.in_features,emb_dim,bias=False)
        )

    def forward(self,feat,pos):
        feat=feat[:,-1:] ## not using background channel, 32*2*224*224->32*1*224*224
        residual=self.shortcut(feat)
        feat=self.feat_conv(feat)
        feat=feat+residual

        fused=self.cross_attn(feat,pos)
        embedding=self.ResNet(fused)
        embedding=nn.functional.normalize(embedding,p=2,dim=-1)
        return embedding

## dataloader

In [12]:
#normal dataset building
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
all_species=['frog','zebrafish']

train_set,val_set,test_set={},{},{}

restrict_classes=None
background_image_path=f'./use_data/Heatmap_Paintings/{mission_name}'
for species in all_species:
    for _set,set_type in [(train_set,'train'),(val_set,'val'),(test_set,'test')]:
        image_path=f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_{species}/{set_type}'
        _set[species]=TiffDataset(root_dir=image_path,background_dir=background_image_path,embedding_type=embedding_type,transform=data_transform[set_type],classes=restrict_classes)
        if set_type=='train':
            restrict_classes=_set[species].classes
        elif set_type=='test':
            restrict_classes=None

classes_of_species={}
for species in all_species:
    classes_of_species[species]=train_set[species].classes

In [13]:
#normal dataloader building
all_species=['frog','zebrafish']

train_loaders,val_loaders,test_loaders=[],[],[]
for species in all_species:
    for _loaders,_set,_type in [(train_loaders,train_set,'train'),(val_loaders,val_set,'val'),(test_loaders,test_set,'test')]:
        _loader=DataLoader(dataset=_set[species],batch_size=32,shuffle=True,drop_last=(_type=='train'))
        _loaders.append((species,_loader))

## train & test

In [12]:
feat_dim=1##
pos_dim=train_set[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
dropout_rate=0
model=DualStreamPositionNet_ablate_V1(feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate).to(device)

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=100
train(model,train_loaders,val_loaders,loss_function,optimizer,device,epochs)

Epoch 1: 100%|██████████████████████████████████████████████████████████| 2250/2250 [35:13<00:00,  1.06it/s, loss=9.11]


epoch 1, loss:9.1112


Epoch 2: 100%|██████████████████████████████████████████████████████████| 2250/2250 [34:46<00:00,  1.08it/s, loss=8.69]


epoch 2, loss:8.6893


Epoch 3: 100%|██████████████████████████████████████████████████████████| 2250/2250 [34:29<00:00,  1.09it/s, loss=8.24]


...

epoch 99, loss:1.6475


Epoch 100: 100%|████████████████████████████████████████████████████████| 2250/2250 [37:59<00:00,  1.01s/it, loss=1.64]


epoch 100, loss:1.6354
At epoch 100 :


Species: frog
accuracy:0.6723, f1_score:0.6597


Species: zebrafish
accuracy:0.7106, f1_score:0.6967


In [13]:
#test 80 epoch model
feat_dim=1##
pos_dim=train_set[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
model=DualStreamPositionNet_ablate_V1(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/ablate_V1_frog_zebrafish_80_epoch_model.pkl",weights_only=True))

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/ablate_V1_frog_zebrafish_80_epoch_loss_func.pkl",weights_only=True))

_=test(model,test_loaders,loss_function,device)

Species: frog
accuracy:0.6599, f1_score:0.6423


Species: zebrafish
accuracy:0.7534, f1_score:0.7502


# Train and Test without using position channel

## changes

In [9]:
class DualStreamPositionNet_ablate_V2(nn.Module):
    def __init__(self,feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate=0):
        super().__init__()
        self.feat_conv=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,7,padding=3),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU()
        )
        self.shortcut=nn.Sequential(
            nn.Conv2d(feat_dim,hid_dim,1),
            nn.BatchNorm2d(hid_dim)
        )

        ##self.cross_attn=CrossAttention(hid_dim,pos_dim)
        self.ResNet=resnet34(hid_dim)        
        self.ResNet.fc=nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(self.ResNet.fc.in_features,emb_dim,bias=False)
        )

    def forward(self,feat,pos):
        residual=self.shortcut(feat)
        feat=self.feat_conv(feat)
        feat=feat+residual

        ##fused=self.cross_attn(feat,pos)
        fused=feat##not using position channel
        embedding=self.ResNet(fused)
        embedding=nn.functional.normalize(embedding,p=2,dim=-1)
        return embedding

## dataloader

In [10]:
# normal dataset building
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
all_species=['frog','zebrafish']

train_set,val_set,test_set={},{},{}

restrict_classes=None
background_image_path=f'./use_data/Heatmap_Paintings/{mission_name}'
for species in all_species:
    for _set,set_type in [(train_set,'train'),(val_set,'val'),(test_set,'test')]:
        image_path=f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_{species}/{set_type}'
        _set[species]=TiffDataset(root_dir=image_path,background_dir=background_image_path,embedding_type=embedding_type,transform=data_transform[set_type],classes=restrict_classes)
        if set_type=='train':
            restrict_classes=_set[species].classes
        elif set_type=='test':
            restrict_classes=None

classes_of_species={}
for species in all_species:
    classes_of_species[species]=train_set[species].classes

In [11]:
# normal dataloader building
all_species=['frog','zebrafish']

train_loaders,val_loaders,test_loaders=[],[],[]
for species in all_species:
    for _loaders,_set,_type in [(train_loaders,train_set,'train'),(val_loaders,val_set,'val'),(test_loaders,test_set,'test')]:
        _loader=DataLoader(dataset=_set[species],batch_size=32,shuffle=True,drop_last=(_type=='train'))
        _loaders.append((species,_loader))

## train & test

In [14]:
feat_dim=train_set[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=-1##not using
hid_dim=64
emb_dim=256
dropout_rate=0
model=DualStreamPositionNet_ablate_V2(feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate).to(device)

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=100
train(model,train_loaders,val_loaders,loss_function,optimizer,device,epochs)

Epoch 1: 100%|██████████████████████████████████████████████████████████| 2250/2250 [30:29<00:00,  1.23it/s, loss=8.95]


epoch 1, loss:8.9486


Epoch 2: 100%|██████████████████████████████████████████████████████████| 2250/2250 [29:42<00:00,  1.26it/s, loss=8.19]


epoch 2, loss:8.1866


Epoch 3: 100%|██████████████████████████████████████████████████████████| 2250/2250 [29:23<00:00,  1.28it/s, loss=7.55]


...

epoch 99, loss:1.1772


Epoch 100: 100%|████████████████████████████████████████████████████████| 2250/2250 [29:35<00:00,  1.27it/s, loss=1.18]


epoch 100, loss:1.1807
At epoch 100 :


Species: frog
accuracy:0.7596, f1_score:0.7538


Species: zebrafish
accuracy:0.8392, f1_score:0.8362


In [14]:
#test 75 epoch model
feat_dim=train_set[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=-1##not using
hid_dim=64
emb_dim=256
model=DualStreamPositionNet_ablate_V2(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/ablate_V2_frog_zebrafish_75_epoch_model.pkl",weights_only=True))

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/ablate_V2_frog_zebrafish_75_epoch_loss_func.pkl",weights_only=True))

_=test(model,test_loaders,loss_function,device)

Species: frog
accuracy:0.7535, f1_score:0.7479


Species: zebrafish
accuracy:0.8434, f1_score:0.8401


# Train and Test without using Gaussian smoothing

## changed dataloader

In [9]:
#no smoothing dataset building
#Images without Gaussian noise are also obtained from paint_cell_image_representations.ipynb.
mission_name='Frog_Zebrafish_2000hv_60000cell_without_gaussian_filter'##
embedding_type='ESM1b'
all_species=['frog','zebrafish']

train_set_V3,val_set_V3,test_set_V3={},{},{}##

restrict_classes=None
background_image_path=f'./use_data/Heatmap_Paintings/{mission_name}'
for species in all_species:
    for _set,set_type in [(train_set_V3,'train'),(val_set_V3,'val'),(test_set_V3,'test')]:
        image_path=f'./use_data/Heatmap_Paintings/{mission_name}/{embedding_type}_{species}/{set_type}'
        _set[species]=TiffDataset(root_dir=image_path,background_dir=background_image_path,embedding_type=embedding_type,transform=data_transform[set_type],classes=restrict_classes)
        if set_type=='train':
            restrict_classes=_set[species].classes
        elif set_type=='test':
            restrict_classes=None

classes_of_species={}
for species in all_species:
    classes_of_species[species]=train_set_V3[species].classes

In [10]:
#no smoothing dataloader building
all_species=['frog','zebrafish']

train_loaders_V3,val_loaders_V3,test_loaders_V3=[],[],[]##
for species in all_species:
    for _loaders,_set,_type in [(train_loaders_V3,train_set_V3,'train'),(val_loaders_V3,val_set_V3,'val'),(test_loaders_V3,test_set_V3,'test')]:
        _loader=DataLoader(dataset=_set[species],batch_size=32,shuffle=True,drop_last=(_type=='train'))
        _loaders.append((species,_loader))

## train & test

In [11]:
feat_dim=train_set_V3[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set_V3[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
dropout_rate=0
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim,dropout_rate).to(device)

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=100
train(model,train_loaders_V3,val_loaders_V3,loss_function,optimizer,device,epochs)

Epoch 1: 100%|█████████████████████████████████████████████████████████████| 2250/2250 [33:29<00:00,  1.12it/s, loss=9]


epoch 1, loss:9.0012


Epoch 2: 100%|██████████████████████████████████████████████████████████| 2250/2250 [33:02<00:00,  1.14it/s, loss=8.21]


epoch 2, loss:8.2139


Epoch 3: 100%|██████████████████████████████████████████████████████████| 2250/2250 [33:08<00:00,  1.13it/s, loss=7.52]


...

epoch 99, loss:1.2322


Epoch 100: 100%|████████████████████████████████████████████████████████| 2250/2250 [50:28<00:00,  1.35s/it, loss=1.22]


epoch 100, loss:1.2221
At epoch 100 :


Species: frog
accuracy:0.7706, f1_score:0.7661


Species: zebrafish
accuracy:0.8413, f1_score:0.8378


In [11]:
#test 95 epoch model
feat_dim=train_set_V3[all_species[0]][0]['feature_channels'].shape[0]#feat_dim=2
pos_dim=train_set_V3[all_species[0]][0]['pos_channels'].shape[0]#pos_dim=32
hid_dim=64
emb_dim=256
model=DualStreamPositionNet(feat_dim,pos_dim,hid_dim,emb_dim).to(device)
model.load_state_dict(torch.load("./model/ablate_V3_frog_zebrafish_95_epoch_model.pkl",weights_only=True))

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=max(classes)+1,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,classes in classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/ablate_V3_frog_zebrafish_95_epoch_loss_func.pkl",weights_only=True))

_=test(model,test_loaders_V3,loss_function,device)

Species: frog
accuracy:0.7588, f1_score:0.7568


Species: zebrafish
accuracy:0.8567, f1_score:0.8543
